# AEO2026 Data Preprocessing

This notebook reads the concatenated JSON records in `AEO2026.txt`, separates category metadata from time-series data, cleans and expands the annual observations, and writes one CSV for each AEO metric code.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

In [ ]:
# Find the project root whether the notebook is run from the project root or Script/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Data' / 'Electricity' / 'AEO2026.txt').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'Data' / 'Electricity'
AEO_PATH = DATA_PATH / 'AEO2026.txt'
OUTPUT_PATH = DATA_PATH / 'AEO2026'
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f'Input:  {AEO_PATH}')
print(f'Output: {OUTPUT_PATH}')

## Load the concatenated JSON records

The source is not a JSON array. It contains complete JSON objects one after another, so `JSONDecoder.raw_decode` is used to read every record safely.

In [ ]:
def load_concatenated_json(path):
    text = Path(path).read_text(encoding='utf-8')
    decoder = json.JSONDecoder()
    objects = []
    idx = 0

    while idx < len(text):
        while idx < len(text) and text[idx].isspace():
            idx += 1
        if idx >= len(text):
            break

        obj, idx = decoder.raw_decode(text, idx)
        objects.append(obj)

    return objects


objects = load_concatenated_json(AEO_PATH)
series_objects = [obj for obj in objects if obj.get('series_id')]
category_objects = [obj for obj in objects if obj.get('category_id')]

print(f'All records:       {len(objects):,}')
print(f'Time series:       {len(series_objects):,}')
print(f'Category metadata: {len(category_objects):,}')

## Clean and save category metadata

In [ ]:
category_df = pd.DataFrame(category_objects).copy()

if not category_df.empty:
    category_df = category_df.drop_duplicates(subset=['category_id'])
    category_df['category_id'] = pd.to_numeric(category_df['category_id'], errors='coerce').astype('Int64')
    category_df['parent_category_id'] = pd.to_numeric(
        category_df['parent_category_id'], errors='coerce'
    ).astype('Int64')
    category_df['childseries_count'] = category_df['childseries'].apply(
        lambda value: len(value) if isinstance(value, list) else 0
    )
    category_df['childseries'] = category_df['childseries'].apply(
        lambda value: json.dumps(value) if isinstance(value, (list, dict)) else value
    )
    category_df = category_df.sort_values(['parent_category_id', 'category_id'], na_position='first')

category_file = OUTPUT_PATH / 'AEO2026.categories.csv'
category_df.to_csv(category_file, index=False)
print(f'Saved {category_df.shape[0]:,} category rows to {category_file.name}')
category_df.head()

## Clean and separate the time series

Each series ID has the form `AEO.2026.<scenario>.<metric details>.<frequency>`. The cleaner derives scenario and metric codes, retains useful source metadata, converts values to numeric data, and sorts the year columns chronologically.

In [ ]:
SCENARIO_NAMES = {
    'AEO2025REF': 'AEO2025 Reference',
    'CB2026': 'Counterfactual Baseline',
    'HM2026': 'High Economic Growth',
    'LM2026': 'Low Economic Growth',
    'HIGHOGS': 'High Oil and Gas Supply',
    'LOWOGS': 'Low Oil and Gas Supply',
    'HIGHELDMD': 'High Electricity Demand',
    'ALTELEC': 'Alternative Electricity',
    'ALTTRNP': 'Alternative Transportation',
    'ELECTRNP': 'Alternative Electricity and Transportation',
    'HIGHZTC': 'High Zero-Carbon Technology Cost',
    'LOWZTC': 'Low Zero-Carbon Technology Cost',
}

METADATA_COLUMNS = [
    'series_id', 'name', 'metric_family', 'scenario_code', 'scenario',
    'metric_code', 'frequency', 'units', 'description', 'start', 'end',
    'last_historical_period', 'last_updated'
]


def clean_series(obj):
    series_id = obj['series_id'].strip()
    parts = series_id.split('.')
    if len(parts) < 5:
        raise ValueError(f'Unexpected series ID: {series_id}')

    scenario_code = parts[2]
    metric_details = parts[3]
    metric_code = metric_details.split('_', 1)[0]
    name = str(obj.get('name') or '').strip()

    row = {
        'series_id': series_id,
        'name': name,
        'metric_family': name.split(':', 1)[0].strip() if name else pd.NA,
        'scenario_code': scenario_code,
        'scenario': SCENARIO_NAMES.get(scenario_code, scenario_code),
        'metric_code': metric_code,
        'frequency': str(obj.get('f') or parts[-1]).strip(),
        'units': obj.get('units'),
        'description': obj.get('description'),
        'start': obj.get('start'),
        'end': obj.get('end'),
        'last_historical_period': obj.get('lastHistoricalPeriod'),
        'last_updated': obj.get('last_updated'),
    }

    for period, value in obj.get('data', []):
        row[str(period).strip()] = pd.to_numeric(value, errors='coerce')

    return row


series_rows = [clean_series(obj) for obj in series_objects]
series_df = pd.DataFrame(series_rows)
series_df = series_df.drop_duplicates(subset=['series_id'], keep='last')

year_columns = sorted(
    (column for column in series_df.columns if str(column).isdigit()),
    key=int,
)
series_df = series_df[METADATA_COLUMNS + year_columns]
series_df = series_df.sort_values(['metric_code', 'scenario_code', 'series_id']).reset_index(drop=True)

print(f'Clean series shape: {series_df.shape}')
print(f'Years: {year_columns[0]}–{year_columns[-1]}')
series_df.head()

## Data-quality checks

In [ ]:
quality_summary = pd.Series({
    'source_series_records': len(series_objects),
    'clean_unique_series': len(series_df),
    'duplicate_series_removed': len(series_objects) - len(series_df),
    'missing_series_ids': int(series_df['series_id'].isna().sum()),
    'missing_units': int(series_df['units'].isna().sum()),
    'scenario_count': int(series_df['scenario_code'].nunique()),
    'metric_code_count': int(series_df['metric_code'].nunique()),
})
display(quality_summary.to_frame('value'))
display(series_df.groupby(['scenario_code', 'metric_code']).size().unstack(fill_value=0))

## Export the separated datasets

The default export creates one file per metric code and keeps all scenarios together for easy comparison. The manifest records every output file and its dimensions.

In [ ]:
manifest_rows = []

for metric_code, metric_df in series_df.groupby('metric_code', sort=True):
    frequency_values = sorted(metric_df['frequency'].dropna().unique())
    frequency_label = frequency_values[0] if len(frequency_values) == 1 else 'MIXED'
    output_file = OUTPUT_PATH / f'AEO2026.{metric_code}.{frequency_label}.csv'
    metric_df.to_csv(output_file, index=False)
    manifest_rows.append({
        'metric_code': metric_code,
        'frequency': frequency_label,
        'rows': len(metric_df),
        'columns': len(metric_df.columns),
        'scenarios': metric_df['scenario_code'].nunique(),
        'file': output_file.name,
    })

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(OUTPUT_PATH / 'AEO2026.manifest.csv', index=False)

print(f'Created {len(manifest_df)} metric files in {OUTPUT_PATH}')
display(manifest_df)

### Optional: export one file per scenario

Uncomment the final line if scenario-level files are also useful.

In [ ]:
def export_by_scenario(df, output_path=OUTPUT_PATH):
    for scenario_code, scenario_df in df.groupby('scenario_code', sort=True):
        output_file = output_path / f'AEO2026.{scenario_code}.A.csv'
        scenario_df.to_csv(output_file, index=False)
        print(f'{output_file.name}: {scenario_df.shape}')

# export_by_scenario(series_df)